In [1]:
# ============================================================
# PHASE 6 — CLEAN MODEL INFERENCE & BACKEND HANDOFF
# CELL 1 — IMPORTS AND PACKAGE SETUP
# ============================================================

import os
import json
import joblib
import pandas as pd
import numpy as np

print("✅ Libraries imported")
print("Pandas:", pd.__version__)
print("NumPy:", np.__version__)

✅ Libraries imported
Pandas: 3.0.5
NumPy: 2.5.3


In [2]:
# ============================================================
# CELL 2 — LOCATE MODEL PACKAGE
# ============================================================

# Check the current working directory
print("Current directory:")
print(os.getcwd())

print("\nFiles/folders here:")
for item in os.listdir():
    print(" -", item)

Current directory:
C:\Users\vatsh\OneDrive\Desktop\ml\meowwww

Files/folders here:
 - .idea
 - .venv
 - 02_baselines.ipynb
 - 02_baselines2.ipynb
 - 5-star edible sushi.csv
 - cost_final_model.joblib
 - cox_feature_columns.joblib
 - cox_model.joblib
 - Edible_superClean_sushi++.csv
 - Edible_superClean_sushi.csv
 - feature_columns.joblib
 - missing_data_report.csv
 - ML_model_files
 - ML_MODEL_PACKAGE.ipynb
 - model_manifest.json
 - model_metadata.json
 - newww.ipynb
 - paimana_train_v1.parquet
 - phase-0.ipynb
 - phase4_checkpoint.joblib
 - Phase_4_Explainability.ipynb
 - Phase_5_Risk_Score_Survival.ipynb
 - Phase_6_Model_Packaging.ipynb
 - predictor.py
 - pyproject.toml
 - schedule_final_model.joblib
 - superReady_sushi.csv
 - sushitime.csv
 - sushi_gaanduuu.csv
 - uv.lock


In [3]:
# ============================================================
# CELL 2 — LOAD SAVED MODEL ARTIFACTS
# ============================================================

# Load trained models
cost_final_model = joblib.load("../models/cost_final_model.joblib")
schedule_final_model = joblib.load("../models/schedule_final_model.joblib")
cox_model = joblib.load("../models/cox_model.joblib")

# Load exact feature lists used during training
feature_columns = joblib.load("../models/feature_columns.joblib")
cox_feature_columns = joblib.load("../models/cox_feature_columns.joblib")

# Load model/package metadata
with open("../models/model_manifest.json", "r") as f:
    package_metadata = json.load(f)

print("✅ All model artifacts loaded successfully")

✅ All model artifacts loaded successfully


In [4]:
# ============================================================
# CELL 3 — VERIFY LOADED MODEL PACKAGE
# ============================================================

print("========== MODEL TYPES ==========")
print("Cost model    :", type(cost_final_model).__name__)
print("Schedule model:", type(schedule_final_model).__name__)
print("Cox model     :", type(cox_model).__name__)

print("\n========== FEATURE COUNTS ==========")
print("XGBoost features:", len(feature_columns))
print("Cox features    :", len(cox_feature_columns))

print("\n========== XGBOOST FEATURES ==========")
for i, col in enumerate(feature_columns, start=1):
    print(f"{i:2d}. {col}")

print("\n========== COX FEATURES ==========")
for i, col in enumerate(cox_feature_columns, start=1):
    print(f"{i:2d}. {col}")

print("\n========== PACKAGE METADATA ==========")
print("Package version:", package_metadata.get("package_version"))
print("Risk tiers     :", package_metadata.get("risk_tiers"))

print("\n✅ MODEL PACKAGE VERIFICATION COMPLETE")

========== MODEL TYPES ==========
Cost model    : Pipeline
Schedule model: Pipeline
Cox model     : CoxPHFitter

========== FEATURE COUNTS ==========
XGBoost features: 47
Cox features    : 10

========== XGBOOST FEATURES ==========
 1. month
 2. year
 3. sector
 4. state
 5. approval_year
 6. project_age
 7. original_duration_months
 8. duration_overrun_months
 9. original_cost_crore
10. revised_cost_crore
11. anticipated_cost_crore
12. current_cost
13. cumulative_expenditure_crore
14. anticipated_delay_from_original
15. remaining_org_months
16. delay_revised_months
17. delay_revisied_months
18. milestones_achieved
19. milestones_total
20. milestone_completion_percentage
21. milestone_data_reliable
22. cost_revision_percentage
23. anticipated_cost_percentage
24. exp_vs_anti_prct
25. exp_vs_org_prct
26. exp_vs_rev_prct
27. project_duration_elapsed_percentage
28. cost_growth_3m
29. cost_growth_6m
30. cost_growth_12m
31. expenditure_growth_3m
32. expenditure_growth_6m
33. expenditure_grow

In [5]:
# ============================================================
# CELL 4 — LOAD FINAL CLEAN DATASET
# ============================================================

DATA_FILE = "../data/5-star edible sushi.csv"

df = pd.read_csv(DATA_FILE)

print("✅ Dataset loaded")
print("Rows   :", df.shape[0])
print("Columns:", df.shape[1])

print("\nFirst 5 columns:")
print(df.columns[:5].tolist())

print("\nLast 5 columns:")
print(df.columns[-5:].tolist())

✅ Dataset loaded
Rows   : 109787
Columns: 52

First 5 columns:
['project_id', 'report_month', 'month', 'year', 'sector']

Last 5 columns:
['agency_project_count_as_of_T', 'sector_overrun_rate', 'target_cost_overrun_12m', 'target_schedule_risk_12m', 'cost_target_valid']


In [6]:
# ============================================================
# CELL 5 — VERIFY REQUIRED INFERENCE FEATURES
# ============================================================

# Check XGBoost features
missing_xgb = [
    col for col in feature_columns
    if col not in df.columns
]

# Check Cox features
missing_cox = [
    col for col in cox_feature_columns
    if col not in df.columns
]

print("========== FEATURE CHECK ==========")

print("Required XGBoost features:", len(feature_columns))
print("Missing XGBoost features :", len(missing_xgb))

if missing_xgb:
    print("\n❌ Missing XGBoost features:")
    for col in missing_xgb:
        print(" -", col)
else:
    print("✅ All 47 XGBoost features are present")

print("\nRequired Cox features:", len(cox_feature_columns))
print("Missing Cox features :", len(missing_cox))

if missing_cox:
    print("\n❌ Missing Cox features:")
    for col in missing_cox:
        print(" -", col)
else:
    print("✅ All 10 Cox features are present")

if not missing_xgb and not missing_cox:
    print("\n🎯 FEATURE CHECK PASSED — READY FOR INFERENCE")
else:
    print("\n⚠️ FEATURE CHECK FAILED — DO NOT RUN PREDICTION YET")

========== FEATURE CHECK ==========
Required XGBoost features: 47
Missing XGBoost features : 0
✅ All 47 XGBoost features are present

Required Cox features: 10
Missing Cox features : 0
✅ All 10 Cox features are present

🎯 FEATURE CHECK PASSED — READY FOR INFERENCE


In [7]:
# ============================================================
# CELL 6 — UNIFIED PREDICTION FUNCTION
# ============================================================

def predict_project_row(project_row):
    """
    Takes one raw project feature row and returns
    all downstream model outputs.
    """

    # --------------------------------------------------------
    # 1. Convert input into one-row DataFrame
    # --------------------------------------------------------
    if isinstance(project_row, pd.Series):
        row = project_row.to_frame().T.copy()

    elif isinstance(project_row, dict):
        row = pd.DataFrame([project_row])

    elif isinstance(project_row, pd.DataFrame):
        row = project_row.copy()

    else:
        raise TypeError(
            "project_row must be a dict, Series, or DataFrame"
        )

    # --------------------------------------------------------
    # 2. Make sure all 47 XGBoost features exist
    # --------------------------------------------------------
    missing_features = [
        col for col in feature_columns
        if col not in row.columns
    ]

    if missing_features:
        raise ValueError(
            f"Missing required features: {missing_features}"
        )

    xgb_row = row[feature_columns].copy()

    # --------------------------------------------------------
    # 3. Cost risk
    # --------------------------------------------------------
    cost_probability = float(
        cost_final_model.predict_proba(xgb_row)[0, 1]
    )

    # --------------------------------------------------------
    # 4. Schedule risk
    # --------------------------------------------------------
    schedule_probability = float(
        schedule_final_model.predict_proba(xgb_row)[0, 1]
    )

    # --------------------------------------------------------
    # 5. Cox risk
    # --------------------------------------------------------
    cox_row = row[cox_feature_columns].copy()

    # Cox model cannot accept missing values.
    # Use the model's training means for missing inputs.
    for col in cox_feature_columns:

        if cox_row[col].isna().any():

            if hasattr(cox_model, "_norm_mean"):
                fill_value = float(
                    cox_model._norm_mean[col]
                )
            else:
                fill_value = 0.0

            cox_row[col] = cox_row[col].fillna(fill_value)

    cox_risk = float(
        cox_model.predict_partial_hazard(cox_row).iloc[0]
    )

    # --------------------------------------------------------
    # 6. Normalize Cox risk approximately to 0-1
    # --------------------------------------------------------
    # Partial hazard itself is not a probability.
    # For the unified score, convert it to a bounded value.
    cox_risk_probability = float(
        cox_risk / (1.0 + cox_risk)
    )

    # --------------------------------------------------------
    # 7. Composite risk score
    # --------------------------------------------------------
    composite_score = (
        0.40 * cost_probability +
        0.40 * schedule_probability +
        0.20 * cox_risk_probability
    ) * 100

    composite_score = float(
        np.clip(composite_score, 0, 100)
    )

    # --------------------------------------------------------
    # 8. Risk tier
    # --------------------------------------------------------
    if composite_score < 25:
        risk_tier = "LOW"

    elif composite_score < 50:
        risk_tier = "WATCH"

    elif composite_score < 75:
        risk_tier = "ELEVATED"

    else:
        risk_tier = "CRITICAL"

    # --------------------------------------------------------
    # 9. Return backend-friendly dictionary
    # --------------------------------------------------------
    return {
        "cost_risk_probability": round(
            cost_probability, 6
        ),

        "schedule_risk_probability": round(
            schedule_probability, 6
        ),

        "cox_risk": round(
            cox_risk, 6
        ),

        "cox_risk_probability": round(
            cox_risk_probability, 6
        ),

        "composite_risk_score": round(
            composite_score, 4
        ),

        "risk_tier": risk_tier,

        "model_version": (
            package_metadata["package_version"]
            if "package_metadata" in globals()
            else "1.0.0"
        )
    }


print("✅ predict_project_row() created successfully")

✅ predict_project_row() created successfully


In [8]:
# ============================================================
# CELL 7 — FIRST REAL PROJECT PREDICTION
# ============================================================

# Take one real row from the final dataset
test_row = df.iloc[0]

print("========== TEST PROJECT ==========")
print("Project ID :", test_row["project_id"])
print("Report month:", test_row["report_month"])

# Run the unified prediction function
prediction = predict_project_row(test_row)

print("\n========== MODEL PREDICTION ==========")

for key, value in prediction.items():
    print(f"{key:30s}: {value}")

print("\n✅ FIRST REAL INFERENCE TEST COMPLETE")

========== TEST PROJECT ==========
Project ID : 120100067
Report month: 2015-04-01

========== MODEL PREDICTION ==========
cost_risk_probability         : 0.006809
schedule_risk_probability     : 0.849275
cox_risk                      : 8.750624
cox_risk_probability          : 0.897442
composite_risk_score          : 52.1922
risk_tier                     : ELEVATED
model_version                 : 1.0.0

✅ FIRST REAL INFERENCE TEST COMPLETE


In [9]:
# ============================================================
# CELL 8 — BATCH INFERENCE TEST
# ============================================================

results = []
errors = []

for idx in range(min(20, len(df))):

    try:
        row = df.iloc[idx]

        prediction = predict_project_row(row)

        results.append({
            "project_id": row["project_id"],
            "report_month": row["report_month"],
            **prediction
        })

    except Exception as e:

        errors.append({
            "row_index": idx,
            "project_id": df.iloc[idx]["project_id"],
            "error": str(e)
        })

results_df = pd.DataFrame(results)

print("========== BATCH INFERENCE TEST ==========")
print("Rows tested :", min(20, len(df)))
print("Successful  :", len(results))
print("Failed      :", len(errors))

if errors:
    print("\n❌ ERRORS:")
    for error in errors:
        print(error)
else:
    print("\n✅ ALL 20 PREDICTIONS SUCCESSFUL")

print("\n========== SAMPLE RESULTS ==========")
display(results_df.head(10))

========== BATCH INFERENCE TEST ==========
Rows tested : 20
Successful  : 20
Failed      : 0

✅ ALL 20 PREDICTIONS SUCCESSFUL

========== SAMPLE RESULTS ==========


,project_id,report_month,cost_risk_probability,schedule_risk_probability,cox_risk,cox_risk_probability,composite_risk_score,risk_tier,model_version
0,120100067,2015-04-01,0.006809,0.849275,8.750624,0.897442,52.1922,ELEVATED,1.0.0
1,120100067,2015-05-01,0.007270,0.821518,8.790662,0.897862,51.1087,ELEVATED,1.0.0
2,120100067,2015-06-01,0.011146,0.806315,8.809506,0.898058,50.6596,ELEVATED,1.0.0
3,120100067,2015-07-01,0.008842,0.855736,8.862302,0.898604,52.5552,ELEVATED,1.0.0
4,120100067,2015-08-01,0.009267,0.817079,9.015293,0.900153,51.0569,ELEVATED,1.0.0
5,120100067,2015-09-01,0.005850,0.731420,3.552058,0.780319,45.0972,WATCH,1.0.0
6,160100231,2015-04-01,0.003835,0.857124,1.090904,0.521738,44.8731,WATCH,1.0.0
7,160100231,2015-05-01,0.012807,0.904054,1.186592,0.542667,47.5278,WATCH,1.0.0
8,160100231,2015-06-01,0.001964,0.668794,1.665517,0.624838,39.3271,WATCH,1.0.0
9,160100231,2015-07-01,0.004144,0.241665,0.652120,0.394717,17.7267,LOW,1.0.0


In [10]:
# ============================================================
# CELL 9 — RISK TIER DISTRIBUTION TEST
# ============================================================

# Run predictions on a sample of real project snapshots
sample_size = min(1000, len(df))
sample_df = df.sample(
    n=sample_size,
    random_state=42
)

tier_results = []

for idx, row in sample_df.iterrows():

    try:
        prediction = predict_project_row(row)

        tier_results.append({
            "project_id": row["project_id"],
            "report_month": row["report_month"],
            "cost_risk_probability": prediction["cost_risk_probability"],
            "schedule_risk_probability": prediction["schedule_risk_probability"],
            "cox_risk_probability": prediction["cox_risk_probability"],
            "composite_risk_score": prediction["composite_risk_score"],
            "risk_tier": prediction["risk_tier"]
        })

    except Exception as e:
        pass

tier_df = pd.DataFrame(tier_results)

print("========== RISK TIER DISTRIBUTION ==========")
print("Predictions generated:", len(tier_df))

print("\nRisk tier counts:")
print(tier_df["risk_tier"].value_counts().sort_index())

print("\nRisk tier percentages:")
print(
    (tier_df["risk_tier"].value_counts(normalize=True) * 100)
    .round(2)
    .sort_index()
)

print("\n========== SCORE RANGE ==========")
print("Minimum score:", tier_df["composite_risk_score"].min())
print("Maximum score:", tier_df["composite_risk_score"].max())
print("Mean score   :", tier_df["composite_risk_score"].mean())

print("\n✅ RISK TIER TEST COMPLETE")

========== RISK TIER DISTRIBUTION ==========
Predictions generated: 1000

Risk tier counts:
risk_tier
CRITICAL     31
ELEVATED    110
LOW         589
WATCH       270
Name: count, dtype: int64

Risk tier percentages:
risk_tier
CRITICAL     3.1
ELEVATED    11.0
LOW         58.9
WATCH       27.0
Name: proportion, dtype: float64

========== SCORE RANGE ==========
Minimum score: 1.7613
Maximum score: 95.9077
Mean score   : 27.457801200000002

✅ RISK TIER TEST COMPLETE


In [11]:
# ============================================================
# CELL 10 — SELECT DEMO PROJECTS FROM EACH RISK TIER
# ============================================================

demo_projects = []

for tier in ["LOW", "WATCH", "ELEVATED", "CRITICAL"]:

    tier_rows = tier_df[tier_df["risk_tier"] == tier]

    selected = tier_rows.sample(
        n=min(3, len(tier_rows)),
        random_state=42
    )

    demo_projects.append(selected)

demo_df = pd.concat(
    demo_projects,
    ignore_index=True
)

# Display useful columns
display(
    demo_df[
        [
            "project_id",
            "report_month",
            "cost_risk_probability",
            "schedule_risk_probability",
            "cox_risk_probability",
            "composite_risk_score",
            "risk_tier"
        ]
    ]
)

print("\n✅ DEMO PROJECT SELECTION COMPLETE")

,project_id,report_month,cost_risk_probability,schedule_risk_probability,cox_risk_probability,composite_risk_score,risk_tier
0,N24001083,2021-01-01,0.004935,0.001499,0.472945,9.7163,LOW
1,N24001087,2021-10-01,0.000412,0.000681,0.403469,8.1131,LOW
2,N06000118,2019-11-01,0.042517,0.088374,0.524736,15.7303,LOW
3,N24000790,2019-03-01,0.580634,0.115543,0.422477,36.2966,WATCH
4,N12000111,2016-09-01,0.007575,0.946045,0.549810,49.1410,WATCH
5,N22000145,2021-03-01,0.035868,0.768591,0.544198,43.0623,WATCH
6,N06000110,2015-04-01,0.008949,0.962517,0.615791,51.1745,ELEVATED
7,N16000236,2021-03-01,0.172767,0.865896,0.519839,51.9433,ELEVATED
8,N16000196,2021-08-01,0.438669,0.887537,0.546167,63.9716,ELEVATED
9,N22000133,2018-09-01,0.948453,0.880639,0.562221,84.4081,CRITICAL



✅ DEMO PROJECT SELECTION COMPLETE


In [12]:
# ============================================================
# CELL 11 — BACKEND-STYLE DICTIONARY TEST
# ============================================================

# Take one real project row
backend_test_row = df.iloc[0]

# Convert the row into a normal Python dictionary
backend_input = backend_test_row.to_dict()

print("========== BACKEND INPUT ==========")
print("Input type:", type(backend_input).__name__)
print("Number of fields:", len(backend_input))

# Run prediction exactly as a backend would
backend_prediction = predict_project_row(backend_input)

print("\n========== BACKEND OUTPUT ==========")

for key, value in backend_prediction.items():
    print(f"{key:30s}: {value}")

print("\n✅ BACKEND-STYLE DICTIONARY TEST COMPLETE")

========== BACKEND INPUT ==========
Input type: dict
Number of fields: 52

========== BACKEND OUTPUT ==========
cost_risk_probability         : 0.006809
schedule_risk_probability     : 0.849275
cox_risk                      : 8.750624
cox_risk_probability          : 0.897442
composite_risk_score          : 52.1922
risk_tier                     : ELEVATED
model_version                 : 1.0.0

✅ BACKEND-STYLE DICTIONARY TEST COMPLETE


In [13]:
# ============================================================
# CELL 12 — CREATE FINAL predictor.py
# ============================================================

predictor_code = r'''
from pathlib import Path
import json
import joblib
import numpy as np
import pandas as pd


# ============================================================
# MODEL PACKAGE LOADING
# ============================================================

BASE_DIR = Path(__file__).resolve().parent
MODELS_DIR = (BASE_DIR / ".." / "models").resolve()

cost_final_model = joblib.load(MODELS_DIR / "cost_final_model.joblib")
schedule_final_model = joblib.load(MODELS_DIR / "schedule_final_model.joblib")
cox_model = joblib.load(MODELS_DIR / "cox_model.joblib")

feature_columns = joblib.load(MODELS_DIR / "feature_columns.joblib")
cox_feature_columns = joblib.load(MODELS_DIR / "cox_feature_columns.joblib")

with open(MODELS_DIR / "model_manifest.json", "r") as f:
    package_metadata = json.load(f)


# ============================================================
# UNIFIED PREDICTION FUNCTION
# ============================================================

def predict_project_row(project_row):
    """
    Takes one raw project feature row and returns
    all downstream model outputs.

    Accepted input:
        - dict
        - pandas Series
        - pandas DataFrame
    """

    # --------------------------------------------------------
    # 1. Convert input into one-row DataFrame
    # --------------------------------------------------------
    if isinstance(project_row, pd.Series):
        row = project_row.to_frame().T.copy()

    elif isinstance(project_row, dict):
        row = pd.DataFrame([project_row])

    elif isinstance(project_row, pd.DataFrame):
        row = project_row.copy()

    else:
        raise TypeError(
            "project_row must be a dict, Series, or DataFrame"
        )

    # --------------------------------------------------------
    # 2. Make sure all 47 XGBoost features exist
    # --------------------------------------------------------
    missing_features = [
        col for col in feature_columns
        if col not in row.columns
    ]

    if missing_features:
        raise ValueError(
            f"Missing required features: {missing_features}"
        )

    xgb_row = row[feature_columns].copy()

    # --------------------------------------------------------
    # 3. Cost risk
    # --------------------------------------------------------
    cost_probability = float(
        cost_final_model.predict_proba(xgb_row)[0, 1]
    )

    # --------------------------------------------------------
    # 4. Schedule risk
    # --------------------------------------------------------
    schedule_probability = float(
        schedule_final_model.predict_proba(xgb_row)[0, 1]
    )

    # --------------------------------------------------------
    # 5. Cox risk
    # --------------------------------------------------------
    cox_row = row[cox_feature_columns].copy()

    # Cox model cannot accept missing values.
    # Use the model's training means for missing inputs.
    for col in cox_feature_columns:

        if cox_row[col].isna().any():

            if hasattr(cox_model, "_norm_mean"):
                fill_value = float(
                    cox_model._norm_mean[col]
                )
            else:
                fill_value = 0.0

            cox_row[col] = cox_row[col].fillna(fill_value)

    cox_risk = float(
        cox_model.predict_partial_hazard(cox_row).iloc[0]
    )

    # --------------------------------------------------------
    # 6. Normalize Cox risk approximately to 0-1
    # --------------------------------------------------------
    cox_risk_probability = float(
        cox_risk / (1.0 + cox_risk)
    )

    # --------------------------------------------------------
    # 7. Composite risk score
    # --------------------------------------------------------
    composite_score = (
        0.40 * cost_probability +
        0.40 * schedule_probability +
        0.20 * cox_risk_probability
    ) * 100

    composite_score = float(
        np.clip(composite_score, 0, 100)
    )

    # --------------------------------------------------------
    # 8. Risk tier
    # --------------------------------------------------------
    if composite_score < 25:
        risk_tier = "LOW"

    elif composite_score < 50:
        risk_tier = "WATCH"

    elif composite_score < 75:
        risk_tier = "ELEVATED"

    else:
        risk_tier = "CRITICAL"

    # --------------------------------------------------------
    # 9. Backend-friendly output
    # --------------------------------------------------------
    return {
        "cost_risk_probability": round(
            cost_probability, 6
        ),

        "schedule_risk_probability": round(
            schedule_probability, 6
        ),

        "cox_risk": round(
            cox_risk, 6
        ),

        "cox_risk_probability": round(
            cox_risk_probability, 6
        ),

        "composite_risk_score": round(
            composite_score, 4
        ),

        "risk_tier": risk_tier,

        "model_version": (
            package_metadata["package_version"]
            if "package_version" in package_metadata
            else "1.0.0"
        )
    }
'''


# Write the file
with open("../inference/predictor.py", "w", encoding="utf-8") as f:
    f.write(predictor_code)

print("✅ predictor.py created successfully")

✅ predictor.py created successfully


In [14]:
    # ============================================================
# CELL 13 — INDEPENDENT predictor.py TEST
# ============================================================

import importlib.util

# Load predictor.py as a completely separate module
spec = importlib.util.spec_from_file_location(
    "backend_predictor",
    os.path.abspath("../inference/predictor.py")
)

backend_predictor = importlib.util.module_from_spec(spec)
spec.loader.exec_module(backend_predictor)

print("========== MODULE CHECK ==========")
print("Module loaded successfully")
print("Function available:",
      hasattr(backend_predictor, "predict_project_row"))

# Use the same real project row as before
independent_input = df.iloc[0].to_dict()

# IMPORTANT:
# Prediction is coming from predictor.py,
# NOT from the notebook's predict_project_row()
independent_prediction = (
    backend_predictor.predict_project_row(independent_input)
)

print("\n========== INDEPENDENT MODULE OUTPUT ==========")

for key, value in independent_prediction.items():
    print(f"{key:30s}: {value}")

print("\n========== COMPARISON ==========")

print(
    "Matches notebook prediction:",
    independent_prediction == prediction
)

print("\n✅ INDEPENDENT predictor.py TEST COMPLETE")

========== MODULE CHECK ==========
Module loaded successfully
Function available: True

========== INDEPENDENT MODULE OUTPUT ==========
cost_risk_probability         : 0.006809
schedule_risk_probability     : 0.849275
cox_risk                      : 8.750624
cox_risk_probability          : 0.897442
composite_risk_score          : 52.1922
risk_tier                     : ELEVATED
model_version                 : 1.0.0

========== COMPARISON ==========
Matches notebook prediction: False

✅ INDEPENDENT predictor.py TEST COMPLETE


In [15]:
# ============================================================
# CELL 14 — DIAGNOSE PREDICTION COMPARISON
# ============================================================

print("========== NOTEBOOK PREDICTION ==========")

for key, value in prediction.items():
    print(
        f"{key:30s} | value={value!r} | type={type(value).__name__}"
    )

print("\n========== predictor.py OUTPUT ==========")

for key, value in independent_prediction.items():
    print(
        f"{key:30s} | value={value!r} | type={type(value).__name__}"
    )

print("\n========== KEY-BY-KEY COMPARISON ==========")

all_keys = set(prediction.keys()) | set(independent_prediction.keys())

for key in sorted(all_keys):

    notebook_value = prediction.get(key)
    module_value = independent_prediction.get(key)

    print(f"\n{key}")
    print("  Notebook :", repr(notebook_value))
    print("  Module   :", repr(module_value))
    print("  Equal    :", notebook_value == module_value)
    print("  Types    :", type(notebook_value).__name__,
          "vs",
          type(module_value).__name__)

print("\n✅ COMPARISON DIAGNOSTIC COMPLETE")

========== NOTEBOOK PREDICTION ==========
cost_risk_probability          | value=0.151886 | type=float
schedule_risk_probability      | value=0.861036 | type=float
cox_risk                       | value=0.783212 | type=float
cox_risk_probability           | value=0.439214 | type=float
composite_risk_score           | value=49.3012 | type=float
risk_tier                      | value='WATCH' | type=str
model_version                  | value='1.0.0' | type=str

========== predictor.py OUTPUT ==========
cost_risk_probability          | value=0.006809 | type=float
schedule_risk_probability      | value=0.849275 | type=float
cox_risk                       | value=8.750624 | type=float
cox_risk_probability           | value=0.897442 | type=float
composite_risk_score           | value=52.1922 | type=float
risk_tier                      | value='ELEVATED' | type=str
model_version                  | value='1.0.0' | type=str

========== KEY-BY-KEY COMPARISON ==========

composite_risk_score
  Not

In [16]:
# ============================================================
# CELL 15 — FINAL EXACT MODULE VERIFICATION
# ============================================================

# Same exact row used in the original test
final_test_row = df.iloc[0]

# Prediction directly from notebook function
notebook_final = predict_project_row(final_test_row)

# Prediction from standalone predictor.py
module_final = backend_predictor.predict_project_row(
    final_test_row.to_dict()
)

print("========== NOTEBOOK ==========")
print(notebook_final)

print("\n========== predictor.py ==========")
print(module_final)

print("\n========== FINAL COMPARISON ==========")
print("Exact match:", notebook_final == module_final)

if notebook_final == module_final:
    print("\n🎉 FINAL INFERENCE VERIFICATION PASSED")
else:
    print("\n❌ FINAL VERIFICATION FAILED")

========== NOTEBOOK ==========
{'cost_risk_probability': 0.006809, 'schedule_risk_probability': 0.849275, 'cox_risk': 8.750624, 'cox_risk_probability': 0.897442, 'composite_risk_score': 52.1922, 'risk_tier': 'ELEVATED', 'model_version': '1.0.0'}

========== predictor.py ==========
{'cost_risk_probability': 0.006809, 'schedule_risk_probability': 0.849275, 'cox_risk': 8.750624, 'cox_risk_probability': 0.897442, 'composite_risk_score': 52.1922, 'risk_tier': 'ELEVATED', 'model_version': '1.0.0'}

========== FINAL COMPARISON ==========
Exact match: True

🎉 FINAL INFERENCE VERIFICATION PASSED
